# Tutorial — Seleção de Features via PySpark (`yggdrasil.feature_selection`)

Seleção de **features** para modelos de classificação/regressão, organizada por **books** (origens de
dados — ex.: serasa, bvs). Para cada book, roda um pipeline de filtros duros (missing, variância,
redundância) + importância (RandomForest + univariadas) + **Boruta**, e consolida tudo num **consenso**
com `selecionada` + `motivo`. No fim: tabela e painéis por book, mais um **ranking global** das mais
fortes para entrar nos modelos. É um subpacote **isolado** — não interfere na esteira de modelo nem na EDA.

> **Spark-first.** O entrypoint recebe um **Spark DataFrame** com o contrato do pacote: features `feat_`,
> alvo `target` e (opcional) coluna de amostra `amostra` (a seleção usa a amostra de desenvolvimento `DES`).
> Requer o extra `[spark]`: `pip install 'yggdrasil[spark]'`.

## 0. Setup e sessão Spark

O notebook roda tanto no **Databricks** (onde `spark` já existe) quanto no **Jupyter local** (cria uma
`SparkSession` local). Em base grande, prefira o cluster e use `sample_size` (ver seção 8).

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path

def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None

_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(0)

# Usa o `spark` do Databricks se existir; senão, cria uma sessão local.
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("ygg-feature-selection-tutorial")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 1. Dataset sintético (books `serasa` e `bvs`)

Montamos uma base com casos pensados para **exercitar cada etapa** do pipeline:

| Feature | Book | Papel esperado |
|---|---|---|
| `feat_serasa_score` | serasa | forte preditora |
| `feat_serasa_atraso` | serasa | **redundante** com o score (|corr|≈0.998) |
| `feat_serasa_pendencias` | serasa | sinal moderado |
| `feat_serasa_consultas` | serasa | ruído (sem sinal) |
| `feat_serasa_const` | serasa | **constante** → sem variância |
| `feat_serasa_vazamento` | serasa | **leakage** (cópia ruidosa do alvo) |
| `feat_bvs_renda` | bvs | forte preditora (sinal oposto) |
| `feat_bvs_util` | bvs | sinal moderado |
| `feat_bvs_score` | bvs | sinal moderado |
| `feat_bvs_ruido` | bvs | ruído |
| `feat_bvs_miss` | bvs | **80% missing** → descartada |

In [ ]:
n = 4000
s_score = rng.normal(size=n)     # serasa: sinal forte
renda   = rng.normal(size=n)     # bvs: sinal forte (oposto)

df = pd.DataFrame({
    # ── book serasa ────────────────────────────────────────────────
    "feat_serasa_score":      s_score,
    "feat_serasa_atraso":     0.97 * s_score + rng.normal(0, 0.05, n),   # redundante c/ score
    "feat_serasa_pendencias": 0.5 * s_score + rng.normal(0, 0.9, n),     # sinal moderado
    "feat_serasa_consultas":  rng.normal(size=n),                        # ruído
    "feat_serasa_const":      1.0,                                       # constante
    # ── book bvs ───────────────────────────────────────────────────
    "feat_bvs_renda":         renda,
    "feat_bvs_util":          0.45 * renda + rng.normal(0, 0.9, n),      # sinal moderado
    "feat_bvs_score":         0.55 * renda + rng.normal(0, 0.9, n),      # sinal moderado
    "feat_bvs_ruido":         rng.normal(size=n),                        # ruído
    "feat_bvs_miss":          rng.normal(size=n),                        # vai virar 80% missing
})

# Alvo binário a partir dos dois sinais fortes.
logit = 1.1 * s_score - 0.9 * renda + rng.normal(0, 0.5, n)
df["target"] = (logit > 0).astype(int)

# Leakage óbvio: cópia ruidosa do alvo (deve ser BARRADA pela flag de leakage).
df["feat_serasa_vazamento"] = df["target"] + rng.normal(0, 0.01, n)

# 80% de missing em feat_bvs_miss (acima do missing_max default 50% -> descarte).
df.loc[df.sample(frac=0.8, random_state=1).index, "feat_bvs_miss"] = np.nan

# Contrato: coluna de amostra (a seleção usa só DES).
df["amostra"] = np.where(rng.random(n) < 0.7, "DES", "OOT")

# Spark DataFrame.
sdf = spark.createDataFrame(df)
print("linhas:", sdf.count(), "| colunas:", len(sdf.columns))
sdf.limit(5).toPandas()

## 2. Configuração

`ColumnConfig` define o **contrato de colunas** (prefixo `feat_`, `target`, `amostra`, `DES`).
`FeatureSelectionConfig` reúne os parâmetros da seleção. Aqui usamos uma config **leve** para o tutorial
rodar rápido (Boruta com poucas iterações, floresta pequena). Os **defaults são mais pesados** — veja a
referência completa no `README.md` do módulo.

In [ ]:
from yggdrasil import ColumnConfig
from yggdrasil.feature_selection import run_feature_selection, FeatureSelectionConfig

cfg = ColumnConfig()   # feature_prefix="feat_", target_col="target", sample_col="amostra", dev_sample="DES"

fs_cfg = FeatureSelectionConfig(
    missing_max=0.50,        # descarta feature com >50% de missing
    corr_high=0.80,          # |corr| acima disso = redundância
    boruta_enable=True,      # liga o Boruta
    boruta_max_iter=12,      # leve p/ o tutorial (default 50)
    rf_n_estimators=60,      # floresta pequena p/ o tutorial (default 100)
    rf_max_depth=5,
    backend="spark",         # pyspark.ml (use "driver" p/ sklearn no driver)
    top_k_book=12,
    top_k_overall=15,
)
fs_cfg

## 3. Quickstart — uma chamada

`run_feature_selection` faz tudo: resolve os books, roda o pipeline por book e devolve um
`FeatureSelectionReport`. Passamos `books=["serasa", "bvs"]` (match por **palavra-chave**, ver seção 5).

> Em Spark local isso pode levar de alguns segundos a ~1 min (várias florestas + Boruta).

In [ ]:
report = run_feature_selection(
    sdf, cfg, fs_cfg,
    books=["serasa", "bvs"],         # dois books por palavra-chave
    problem_type="classification",   # explícito (poderia ser inferido do alvo)
    with_panels=True,                # monta os painéis gráficos
)

# Resumo por book: quantas features entraram, foram selecionadas e descartadas.
report.summary()

## 4. Lendo o relatório

O `FeatureSelectionReport` traz:
`selection_table` (1 linha/feature, todas as colunas), `book_tables` (por book), `selected_features`
(dict por book), `selected_overall` (lista achatada), `overall_importance` (ranking global recalculado
sobre as selecionadas), `panels` (figuras) e `problem_type`/`cfg`.

In [ ]:
print("Selecionadas por book:")
for book, feats in report.selected_features.items():
    print(f"  {book}: {feats}")

print("\nTotal selecionado (overall):", report.selected_overall)

In [ ]:
# Tabela de seleção (colunas-chave). 'motivo' explica cada decisão.
cols = ["book", "feature", "pct_missing", "sem_variancia", "score",
        "boruta_decisao", "score_consenso", "selecionada", "motivo"]
report.selection_table[cols]

In [ ]:
# Ranking GLOBAL das selecionadas (recalculado sobre todas as selecionadas juntas).
report.overall_importance.head(15)

## 5. Books — as 3 formas de definir

`books` aceita três formas. Use `resolve_books` para inspecionar o agrupamento **antes** de rodar a
seleção (ele só lê os nomes de coluna).

In [ ]:
from yggdrasil.feature_selection import resolve_books

print("(1) Lista de palavras-chave (match 'contains', case-insensitive):")
for b in resolve_books(sdf, cfg, ["serasa", "bvs"]):
    print(f"    {b.name}: {b.features}")

print("\n(2) Auto (books=None) — 1º segmento após o prefixo 'feat_':")
for b in resolve_books(sdf, cfg, None):
    print(f"    {b.name}: {len(b.features)} features")

print("\n(3) Dict explícito — colunas escolhidas a dedo:")
books_dict = {"bureau": ["feat_serasa_score", "feat_bvs_renda"]}
for b in resolve_books(sdf, cfg, books_dict):
    print(f"    {b.name}: {b.features}")

## 6. O pipeline por book e os `motivo`

A ordem por book é: **missing → variância (P1==P99) → importância → redundância → Boruta → consenso**.
Cada descarte registra um `motivo`.

Descartes **determinísticos** (filtros duros):

- `feat_serasa_const` → **`sem variância`** (P1==P99);
- `feat_bvs_miss` → **`alto missing`** (80% > 50%);
- `feat_serasa_vazamento` → **`suspeita de leakage (revisar)`** (não entra, mesmo com importância altíssima).

Redundância (**não-determinística**): `feat_serasa_score` e `feat_serasa_atraso` caem no **mesmo cluster**
de redundância (|corr|≈0.998 > `corr_high`=0.80). O pipeline mantém só **uma** como representante (a de
maior `score` de importância) e marca a outra como `redundante c/ ...` — qual das duas sobrevive pode
**variar entre execuções**, pois são quase equivalentes.

In [ ]:
sel = report.selection_table.set_index("feature")
casos = ["feat_serasa_const", "feat_bvs_miss", "feat_serasa_vazamento",
         "feat_serasa_score", "feat_serasa_atraso"]   # par redundante: uma vira representante, a outra sai
sel.loc[[f for f in casos if f in sel.index], ["book", "selecionada", "motivo"]]

In [ ]:
# Distribuição dos motivos (quem saiu e por quê).
report.selection_table["motivo"].value_counts(dropna=False)

## 7. Gráficos

Com `with_panels=True`, `report.panels` traz figuras matplotlib indexadas por chave:
`"overview"`, `"overall_importance"`, `"book::<nome>"` e `"corr::<nome>"` (heatmap, só quando o book tem
≥2 numéricas). Cada figura renderiza inline (último expressão da célula).

In [ ]:
report.panels["overview"]            # selecionadas x descartadas por book

In [ ]:
report.panels["overall_importance"] # top global — as mais fortes p/ os modelos

In [ ]:
report.panels["book::serasa"]       # seleção detalhada do book serasa (motivo anotado)

In [ ]:
# Heatmap de correlação do book (pode não existir se houver poucas numéricas).
fig = report.panels.get("corr::serasa")
fig if fig is not None else "sem heatmap para este book"

## 8. Ajustes: Boruta, backend e amostragem

- **`boruta_enable=False`** desliga a etapa mais cara (a seleção passa a se apoiar em importância +
  consenso). Útil para iterar rápido.
- **`backend="driver"`** roda o RandomForest/Boruta com `sklearn` no driver (amostra via `toPandas()`) —
  bom para bases pequenas; **`"spark"`** escala no cluster.
- **`sample_size > 0`** amostra `N` linhas **só para as etapas de modelo** (RF e Boruta); a correlação e
  as univariadas usam sempre o dataset completo.

Exemplo: mesma base, sem Boruta, para comparar o nº de selecionadas.

In [ ]:
fs_rapido = FeatureSelectionConfig(boruta_enable=False, rf_n_estimators=60, rf_max_depth=5,
                                   top_k_book=12, top_k_overall=15)
rep2 = run_feature_selection(sdf, cfg, fs_rapido, books=["serasa", "bvs"],
                             problem_type="classification", with_panels=False)
print("Com Boruta:  ", report.selected_overall)
print("Sem Boruta:  ", rep2.selected_overall)

## 9. Exportar e logar

`to_csv` grava a `selection_table`; `to_html(embed_panels=True)` gera um relatório com os painéis
embutidos (PNG base64). Para registrar no MLflow, passe `mlflow_experiment=...` em
`run_feature_selection` (ver célula comentada).

> **Databricks:** os caminhos abaixo são relativos ao diretório de trabalho do driver. No Databricks,
> aponte para o DBFS/Volumes (ex.: `/dbfs/tmp/selecao_features.csv`); no Jupyter local, o diretório
> atual já serve.

In [ ]:
report.to_csv("selecao_features.csv")
with open("relatorio_selecao.html", "w", encoding="utf-8") as fh:
    fh.write(report.to_html(embed_panels=True))
print("Arquivos gravados: selecao_features.csv, relatorio_selecao.html")

In [ ]:
# MLflow (descomente em um ambiente com tracking configurado, ex.: Databricks):
# report_mlf = run_feature_selection(
#     sdf, cfg, fs_cfg, books=["serasa", "bvs"],
#     mlflow_experiment="/Shared/feature_selection",   # loga selection_table, html, painéis e métricas
#     run_name="fs_serasa_bvs_v1",
# )

## 10. Notas

- **Numeric-only:** RF, correlação, IV/KS/AUC/Gini e Boruta operam só em colunas **numéricas**; features
  não-numéricas ficam `NaN` nesses indicadores.
- **Classificação × regressão:** `iv/ks/auc/gini` existem **só em classificação**; em regressão o
  `score`/consenso se apoiam em `rf_importance` e `corr_target`.
- **Consenso:** `leakage_flag` **barra** a seleção (prioridade máxima); Boruta `confirmada` seleciona;
  senão a feature entra pelo `score_consenso >= consensus_threshold`. **Boruta `rejeitada` não impede** a
  seleção — se o consenso atingir o limiar, ela entra mesmo assim (motivo `selecionada (consenso; Boruta
  rejeitou)`); o Boruta só derruba quando o consenso também fica abaixo do limiar. Pesos e limiares são
  configuráveis.
- **Referência completa** de parâmetros e comportamento: `yggdrasil/feature_selection/README.md`.
- Outros tutoriais: `00_tutorial_yggdrasil.ipynb` (PD), `01_tutorial_lgd.ipynb` (LGD),
  `02_tutorial_eda_features.ipynb` (EDA).